In [ ]:
using DataFrames, CSV, XLSX, Statistics

In [ ]:
using Turing
using DifferentialEquations
using StatsPlots
using LinearAlgebra
import NaNMath

In [ ]:
using DataFrames, XLSX

In [ ]:
med4 = DataFrame(XLSX.readtable("PSSP7_ZT145.xlsx", "Host"))
rename!(med4, [:time, :rep1, :rep2])
t_obs_pro = med4.time
obsdata_pro = (med4.rep1 .+ med4.rep2) ./ 2

virus = DataFrame(XLSX.readtable("PSSP7_ZT145.xlsx", "Virus"))
rename!(virus, [:time, :rep1, :rep2])
t_obs_virus = virus.time
obsdata_virus = (virus.rep1 .+ virus.rep2) ./ 2
nothing

In [ ]:
function pro_virus_basic(du, u, p, t)
    # state variables
    Su = u[1] # susceptible host cells
    Ex = u[2] # exposed host cells
    In = u[3] # infected host cells and the capsid-protected progeny phages appear
    Vi = u[4] # virus
   
    

    # parameters
    μmax = p[1] # maximum growth rate at Lopt (h⁻¹)
    Lopt = p[2] # optimal light (μmol s⁻¹ m⁻²)
    α    = p[3] # initial slope of the light response curve (h⁻¹)
    KL   = p[4] # minimum amount of light necessary for cell division (μmol s⁻¹ m⁻²)
    ω    = p[5] # hose basal mortality (h⁻¹)
    K    = p[6] # host carrying capacity (cells ml⁻¹)
    ϕ    = p[7] # adsorption rate (ml h⁻¹)
    β    = p[8] # burst size (unitless)
    λ   = p[9] # average eclise period (h)(the time between phage attachemnt and the appearence of capsid-protected phages)
    δ    = p[10] # viral decay rate (h⁻¹)
   
    
    
    # light -- hourly data
    n_light = 14; n_dark = 10
    L = 35 # μmol s⁻¹ m⁻²
    # t in hours
    τ = rem(t, 24) # time of day
    
    # light dependent growth rate
    μopt = μmax * L / (L + μmax / α * (L / Lopt - 1.0)^2)
    Lt = L * τ * (1.0 - isless(n_light, τ)) + L * (n_light - (τ - n_dark)) * n_light / n_dark * isless(n_light, τ)
    μ = μopt * Lt^4 / (Lt^4 + KL^4)
    

    total_cells = Su + Ex + In 
    carrying_capacity_factor = max(1.0 - total_cells/K, 0.0)
    

    du[1] = μ * Su * carrying_capacity_factor - ω * Su - ϕ * Su * Vi           # dSu/dt
    du[2] = ϕ * Su * Vi - ω * Ex - Ex / (λ*Vi/total_cells)                                    # dEx/dt
    du[3] = Ex / (λ*Vi/total_cells)   - ω * In - In / (λ*Vi/total_cells)                                      # dIn/dt 
    du[4] = β * In / (λ*Vi/total_cells)   - ϕ * Su * Vi - δ * Vi                                 # dVi/dt
    

   return nothing
end

#export resistance_ratio, pro_virus_basic 

In [ ]:
using Random
Random.seed!(42)  # 设置随机种子，保证可重复性

# ============================================================
# 首先测试原始参数是否能成功求解
# ============================================================
println("=" ^ 60)
println("测试原始参数...")
u0_test = [obsdata_pro[1], obsdata_pro[1]*1e-6, obsdata_pro[1]*1e-6, obsdata_virus[1]]
# 原始参数: [μmax, Lopt, α, KL, ω, K, ϕ, β, λ, δ]
p_original = [0.025, 45.78, 6.1e-4, 255.0, 0.0015, 2.8e9, 0.2e-8, 200.0, 65.0, 6e-4]
tspan_test = (14.0, 136.0)
prob_test = ODEProblem(pro_virus_basic, u0_test, tspan_test, p_original)
sol_test = solve(prob_test, Tsit5(); saveat=1.0, abstol=1e-8, reltol=1e-8)
println("原始参数求解状态: $(sol_test.retcode)")
println("=" ^ 60)

# ============================================================
# 定义参数范围 (基于原始参数的 ±200% 范围，扩大搜索空间)
# ============================================================
# 原始参数: [0.025, 45.78, 6.1e-4, 255.0, 0.0015, 2.8e9, 0.2e-8, 200.0, 65, 6e-4]
param_ranges = Dict(
    :μmax => (0.00833, 0.075),      # 最大生长率 (h⁻¹) - 原值 0.025 (±200%)
    :Lopt => (15.26, 137.34),       # 最适光强 - 原值 45.78 (±200%)
    :α    => (2.03e-4, 1.83e-3),    # 光响应曲线初始斜率 - 原值 6.1e-4 (±200%)
    :KL   => (85.0, 765.0),         # 最低光强 - 原值 255.0 (±200%)
    :ω    => (0.0005, 0.0045),      # 宿主基础死亡率 - 原值 0.0015 (±200%)
    :K    => (9.33e8, 8.4e9),       # 环境容纳量 - 原值 2.8e9 (±200%)
    :ϕ    => (6.67e-10, 6e-9),      # 吸附速率 - 原值 2e-9 (0.2e-8) (±200%)
    :β    => (66.67, 600.0),        # 裂解量 - 原值 200 (±200%)
    :λ    => (21.67, 195.0),        # 潜伏期基础值 (h) - 原值 65 (±200%)
    :δ    => (2e-4, 1.8e-3)         # 病毒衰减率 - 原值 6e-4 (±200%)
)

# 参数名称顺序（与模型中 p 向量的顺序一致）
param_names = [:μmax, :Lopt, :α, :KL, :ω, :K, :ϕ, :β, :λ, :δ]

# ============================================================
# 生成随机参数组合
# ============================================================
n_samples = 10000  # 生成 10000 组随机参数

# 从均匀分布中随机采样
function sample_parameters(ranges, names, n)
    params = []
    for i in 1:n
        p = [rand() * (ranges[name][2] - ranges[name][1]) + ranges[name][1] for name in names]
        push!(params, p)
    end
    return params
end

# 生成参数集合
param_sets = sample_parameters(param_ranges, param_names, n_samples)

println("生成了 $(n_samples) 组随机参数组合")
println("\n参数范围：")
for name in param_names
    println("  $(name): $(param_ranges[name])")
end

# 显示前 5 组参数示例
println("\n前 5 组参数示例：")
for i in 1:min(5, n_samples)
    println("  参数组 $i: ", round.(param_sets[i], sigdigits=3))
end

# ============================================================
# 初始条件和时间范围
# ============================================================
# 4个状态变量: [Su, Ex, In, Vi]
u0 = [obsdata_pro[1], obsdata_pro[1]*1e-6, obsdata_pro[1]*1e-6, obsdata_virus[1]]
tspan = (14.0, 136.0)

# ============================================================
# 定义拟合优度评估函数
# ============================================================
"""
计算模型预测与观测数据的拟合优度
返回: (RMSE_host, RMSE_virus, R²_host, R²_virus, total_score)
"""
function evaluate_fit(sol, t_obs_host, obs_host, t_obs_virus, obs_virus)
    # 在观测时间点插值获取模型预测值
    # 宿主：总细胞数 = Su + Ex + In
    pred_host = [sol(t)[1] + sol(t)[2] + sol(t)[3] for t in t_obs_host]
    # 病毒
    pred_virus = [sol(t)[4] for t in t_obs_virus]
    
    # 计算 RMSE (均方根误差)
    rmse_host = sqrt(mean((pred_host .- obs_host).^2))
    rmse_virus_log = sqrt(mean((log10.(max.(pred_virus, 1.0)) .- log10.(max.(obs_virus, 1.0))).^2))
    
    # 计算 R² (决定系数)
    ss_res_host = sum((pred_host .- obs_host).^2)
    ss_tot_host = sum((obs_host .- mean(obs_host)).^2)
    r2_host = 1 - ss_res_host / ss_tot_host
    
    ss_res_virus = sum((log10.(max.(pred_virus, 1.0)) .- log10.(max.(obs_virus, 1.0))).^2)
    ss_tot_virus = sum((log10.(max.(obs_virus, 1.0)) .- mean(log10.(max.(obs_virus, 1.0)))).^2)
    r2_virus = 1 - ss_res_virus / ss_tot_virus
    
    # 归一化 RMSE (相对于观测数据的标准差)
    nrmse_host = rmse_host / std(obs_host)
    nrmse_virus = rmse_virus_log / std(log10.(max.(obs_virus, 1.0)))
    
    # 综合评分 (越小越好)
    total_score = 0.5 * nrmse_host + 0.5 * nrmse_virus
    
    return (rmse_host=rmse_host, rmse_virus=rmse_virus_log, 
            r2_host=r2_host, r2_virus=r2_virus,
            nrmse_host=nrmse_host, nrmse_virus=nrmse_virus,
            total_score=total_score)
end

# ============================================================
# 设置筛选标准
# ============================================================
R2_THRESHOLD_HOST = 0.5      # 宿主 R² 至少 0.5
R2_THRESHOLD_VIRUS = 0.5     # 病毒 R² 至少 0.5
NRMSE_THRESHOLD = 1.0        # NRMSE < 1 表示误差小于数据标准差
TOP_PERCENT = 10             # 保留最好的 10%

println("\n筛选标准：")
println("  R² 阈值 (宿主): ≥ $(R2_THRESHOLD_HOST)")
println("  R² 阈值 (病毒): ≥ $(R2_THRESHOLD_VIRUS)")
println("  NRMSE 阈值: ≤ $(NRMSE_THRESHOLD)")
println("  保留最好的: $(TOP_PERCENT)%")

# ============================================================
# 对所有参数组合求解 ODE 并评估
# ============================================================
all_solutions = []
successful_params = []
fit_scores = []
failed_reasons = Dict{Symbol, Int}()

println("\n开始求解和评估...")

for (i, p) in enumerate(param_sets)
    prob = ODEProblem(pro_virus_basic, u0, tspan, p)
    try
        sol = solve(prob, Tsit5(); 
                    saveat=0.1,
                    abstol=1e-6, 
                    reltol=1e-6,
                    maxiters=1e7)
        
        if sol.retcode == ReturnCode.Success
            # 评估拟合优度
            fit_result = evaluate_fit(sol, t_obs_pro, obsdata_pro, t_obs_virus, obsdata_virus)
            
            push!(all_solutions, sol)
            push!(successful_params, p)
            push!(fit_scores, fit_result)
        else
            reason = Symbol(sol.retcode)
            failed_reasons[reason] = get(failed_reasons, reason, 0) + 1
        end
    catch e
        failed_reasons[:Exception] = get(failed_reasons, :Exception, 0) + 1
    end
    
    # 进度显示
    if i % 1000 == 0
        println("  已处理: $i / $n_samples")
    end
end

println("\n成功求解: $(length(all_solutions)) / $(n_samples) 组参数")

# 显示失败原因
if !isempty(failed_reasons)
    println("\n失败原因统计：")
    for (reason, count) in failed_reasons
        println("  $(reason): $(count) 次")
    end
end

# ============================================================
# 根据标准筛选参数
# ============================================================
good_by_r2 = findall(s -> s.r2_host >= R2_THRESHOLD_HOST && s.r2_virus >= R2_THRESHOLD_VIRUS, fit_scores)
good_by_nrmse = findall(s -> s.nrmse_host <= NRMSE_THRESHOLD && s.nrmse_virus <= NRMSE_THRESHOLD, fit_scores)
n_top = max(1, Int(ceil(length(fit_scores) * TOP_PERCENT / 100)))
sorted_indices = sortperm([s.total_score for s in fit_scores])
good_by_top = length(fit_scores) > 0 ? sorted_indices[1:min(n_top, length(sorted_indices))] : Int[]
good_indices = intersect(good_by_r2, good_by_nrmse)

println("\n筛选结果：")
println("  通过 R² 阈值: $(length(good_by_r2)) 组")
println("  通过 NRMSE 阈值: $(length(good_by_nrmse)) 组")
println("  最好的 $(TOP_PERCENT)%: $(length(good_by_top)) 组")
println("  综合通过: $(length(good_indices)) 组")

# ============================================================
# 显示最佳参数组合
# ============================================================
if length(fit_scores) > 0
    best_idx = sorted_indices[1]
    best_params = successful_params[best_idx]
    best_score = fit_scores[best_idx]
    
    println("\n" * "=" ^ 60)
    println("最佳参数组合：")
    println("=" ^ 60)
    for (i, name) in enumerate(param_names)
        println("  $(name) = $(round(best_params[i], sigdigits=4))")
    end
    println("\n拟合优度：")
    println("  R² (宿主): $(round(best_score.r2_host, digits=4))")
    println("  R² (病毒): $(round(best_score.r2_virus, digits=4))")
    println("  NRMSE (宿主): $(round(best_score.nrmse_host, digits=4))")
    println("  NRMSE (病毒): $(round(best_score.nrmse_virus, digits=4))")
    println("  综合评分: $(round(best_score.total_score, digits=4))")
end

# ============================================================
# 绑图：显示筛选后的结果
# ============================================================
# 使用所有通过筛选标准的参数组合绑图
plot_indices = good_indices

p1 = plot(
    xlabel="Time (h)",
    ylabel="Cell count (cells/mL)",
    legend=:outertopright,
    left_margin=8Plots.mm,
    bottom_margin=6Plots.mm,
    title="Host Cells - All Passed ($(length(plot_indices)) sets)"
)

p2 = plot(
    xlabel="Time (h)",
    ylabel="Virus count (copies/mL)",
    yscale=:log10,
    legend=:outertopright,
    left_margin=8Plots.mm,
    bottom_margin=6Plots.mm,
    title="Virus - All Passed ($(length(plot_indices)) sets)"
)

# 绑制筛选后的曲线
for idx in plot_indices
    sol = all_solutions[idx]
    sol_array = Array(sol)'
    total_host = sol_array[:,1] .+ sol_array[:,2] .+ sol_array[:,3]
    plot!(p1, sol.t, total_host, alpha=0.3, color=:blue, label="")
    plot!(p2, sol.t, sol_array[:,4], alpha=0.3, color=:red, label="")
end

# 绑制最佳拟合曲线（加粗）
if length(fit_scores) > 0
    best_sol = all_solutions[best_idx]
    best_array = Array(best_sol)'
    best_host = best_array[:,1] .+ best_array[:,2] .+ best_array[:,3]
    plot!(p1, best_sol.t, best_host, linewidth=3, color=:darkblue, label="Best fit")
    plot!(p2, best_sol.t, best_array[:,4], linewidth=3, color=:darkred, label="Best fit")
end

# 添加观测数据点
scatter!(p1, t_obs_pro, obsdata_pro, label="Host data", color=:black, markersize=6)
scatter!(p2, t_obs_virus, obsdata_virus, label="Virus data", color=:black, markersize=6)

# 组合图
plot(p1, p2, size=(1200, 500), layout=(1, 2))